<a href="https://colab.research.google.com/github/lshofsl/EngramNCA/blob/main/GeneNCA_RA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **EngramNCA**

In [ ]:
#@title Get Packages and Images  { vertical-output: true}

!pip install git+https://github.com/lshofsl/EngramNCA
!git clone --depth 1 --filter=blob:none --sparse https://github.com/lshofsl/EngramNCA.git
!cd EngramNCA/ && git sparse-checkout set Images
!mkdir Images && mv EngramNCA/Images/* Images && rm -rf EngramNCA

In [ ]:
#@title Imports { vertical-output: true}
import random
import numpy as np
import matplotlib.pyplot as plt
import os
import torch

os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

from NCA.NCA_gene import *
import NCA.utils as utils
from IPython.display import Image, HTML, clear_output
import torch.nn.functional as F
import logging
from IPython.display import display, HTML, Video
from PIL import Image
import cv2
from base64 import b64encode
logger = logging.getLogger()
old_level = logger.level
logger.setLevel(100)




In [ ]:
#@title Setup { vertical-output: true}
HEIGHT = 30 #@param {type:"integer"}
WIDTH = 30 #@param {type:"integer"}
CHANNELS = 22 # @param {type:"integer"}<--- NCA feature channels
BATCH_SIZE = 12 #@param {type:"integer"}
PADDING = 5 #@param {type:"integer"}
GENE_COUNT = 3 #@param {type:"integer"} <-- Number of gene channels to use for "private" information
RECURRENT_CHANNELS = 3 #@param {type:"integer"} <-- Number of channels for RA computation, these channels are not updated by the NCA and are only used for recurrent processing
MODULATORY_OUTPUT = 3 #@param {type:"integer"} <-- Number of channels for modulatory output, these channels are not updated  and only show the output of the NCA
POOL_SIZE = 2666 #@param {type:"integer"}<--- NCA training pool size, lower values train faster but are less stable
TRAINING_ITERS = 8000  #@param {type:"integer"}<-- Number of trainign iterations
HIDDEN_SIZE = 64 #@param {type:"integer"}<--- NCA hidden size
PRIMITIVES_SHAPES = ["Images/square.png", "Images/circle.png", "Images/triangle.png"]
PRIMITIVES_BODY_PARTS = ["Images/Torso.png", "Images/Head.png", "Images/Tail.png", "Images/leg1.png", "Images/leg2.png", "Images/leg3.png", 'Images/leg4.png']
PRIMITIVES_LINES = ["Images/horizontal.png", "Images/Verical.png"]
style = """
<style>
.output_wrapper, .output {
    display: flex;
    flex-direction: row-reverse; /* Align content to the right */
}
</style>
"""


In [ ]:
#@title Load Primitives { vertical-output: true}

paths = PRIMITIVES_SHAPES #@param {type:"string"}
images = []
images_to_display = []
for path in paths:
    image, image_to_display = utils.get_image(path, HEIGHT, WIDTH, padding=PADDING)
    images.append(image)
    images_to_display.append(image_to_display)

genes = [[0], [1], [2]] # <-- Gene one hot encoding, indicates which bits if the gene sequence for each encoded "image" should be 1, [0] = 001, [0,1] = 011, [2] = 100 etc. for 3 bits genes. One, one-hot encoding per image, this rule applies for any gene size

HEIGHT = HEIGHT + 2*PADDING
WIDTH = WIDTH + 2*PADDING
assert len(paths) == len(genes), 'Genes and images should have the same length '

In [ ]:
#@title Display Primitives { vertical-output: true}
for i,image in enumerate(images_to_display):
    plt.figure(3+i)
    plt.imshow(image)
pools = []
for gene in genes:
    pools.append(utils.make_gene_pool_GeneCA(gene, pool_size=POOL_SIZE,height=HEIGHT, width=WIDTH, channels=CHANNELS, gene_size=GENE_COUNT, device=DEVICE))
seeds = []
for pool in pools:
    seeds.append(pool[0].clone())

In [ ]:
#@title Get Batch Image Partitions { vertical-output: true}
partitions = len(paths)
if partitions == 1:
    part = [BATCH_SIZE]
div = BATCH_SIZE//partitions
rem = BATCH_SIZE % partitions
part = [div + 1 if i < rem else div for i in range(partitions)]
print(f"Batch image paritions = {part}. Batch Size of {BATCH_SIZE}. Number of Partitions = {partitions}")

In [ ]:
#@title Load Filters for Loss Function { vertical-output: true}
sobel_x = torch.tensor([[-1.0, 0.0, 1.0], [-2.0, 0.0, 2.0], [-1.0, 0.0, 1.0]], dtype=torch.float32, device=DEVICE)
lap = torch.tensor([[1.0, 2.0, 1.0], [2.0, -12, 2.0], [1.0, 2.0, 1.0]], dtype=torch.float32, device=DEVICE)
filters = torch.stack([sobel_x, sobel_x.T, lap])
folder = "Gene"

In [ ]:
#@title Create Path for Saving Models { vertical-output: true}
path = "Trained_models/" + folder
if not os.path.exists(path):
    os.makedirs(path)
    print(f"Path: {path} created")
else:
    print(f"Path: {path} already exists, all OK!")


In [ ]:
#@title Initialise NCA { vertical-output: true}
bases = [images[i].tile(part[i],1,1,1) for i in range(len(part))]
base = torch.cat(bases, dim =0 ).to(DEVICE)
loss_log = []
nca = GeneCA(CHANNELS,HIDDEN_SIZE, gene_size=GENE_COUNT)
nca = nca.to(DEVICE)
optim = torch.optim.AdamW(nca.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.StepLR(optim, step_size=2000, gamma=0.3)
name = folder + "/" +type(nca).__name__ + "_gene_size_" +str(GENE_COUNT)

In [ ]:
#@title Training { vertical-output: true}

lambda_p = 0.1
lambda_m = 0.1

alpha_log =  []
beta_log   = []
omega_log  = []
K_log      = []
kappa_log  = []
log_iters  = []
a          = []
b          = []
d          = []
first_hiddens = []

for i in range(TRAINING_ITERS + 1):
    with torch.no_grad():
        idxs, x = utils.get_gene_pool(pools, part, seeds)

    # Ensure x is ready for gradients
    x = x.detach().clone()
    x.requires_grad = True

    optim.zero_grad()

    # Initialize loss tensors on the correct device
    total_amp_loss = torch.tensor(0.0, device=x.device)
    total_smooth_loss = torch.tensor(0.0, device=x.device)

    # Use a local list to track states if needed, but avoid modifying 'x' in-place
    current_x = x

    itters = random.randrange(32, 92)
    for t in range(itters):
        m_prev = current_x[:, 19:22].clone()

        out, phase, amplitud = nca(current_x, step=t, k=4)
        current_x = out

        total_amp_loss += amplitud.pow(2).mean()
        total_smooth_loss += (current_x[:, 19:22] - m_prev).pow(2).mean()

    # Image loss using the final state
    x_rgb = current_x[:, :4, :, :]
    loss_img = (base - x_rgb).pow(2).mean() + \
               0.1 * (perchannel_conv(base, filters) - perchannel_conv(x_rgb, filters)).pow(2).mean()

    loss = loss_img + (lambda_p * total_amp_loss / itters) + (lambda_m * total_smooth_loss / itters)

    loss.backward()

    # Gradient clipping (Stabilizes NCA training)
    for p in nca.parameters():
        if p.grad is not None:
            p.grad /= (p.grad.norm() + 1e-8)

    optim.step()

    # 6. Housekeeping (Use no_grad for memory management)
    loss_log.append(loss.log().item())
    with torch.no_grad():
        # Update pool with detached state so we don't carry the graph
        pools = utils.udate_gene_pool(pools, current_x.detach(), idxs, part)

    scheduler.step()

    if i % 100 == 0:
        print(f"Training itter {i}, loss = {loss.item()}")
        plt.clf()
        clear_output()
        plt.figure(1,figsize=(10, 4))
        plt.title('Loss history')
        print(name)
        plt.plot(loss_log, '.', alpha=0.5, color = "b")
        utils.show_batch(x[2:10])
        plt.show(block=False)
        plt.pause(0.01)

        alpha_log.append(nca.alpha.item())
        beta_log.append(nca.beta.item())
        omega_log.append(nca.omega.item())
        K_log.append(nca.K.item())
        kappa_log.append(nca.kappa.item())
        log_iters.append(i)
        a.append(current_x[:, 16:17].detach().cpu().numpy())
        b.append(current_x[:, 17:18].detach().cpu().numpy())
        d.append(current_x[:, 18:19].detach().cpu().numpy())
        first_hiddens.append(current_x[:, 4:8].detach().cpu().numpy())


    if i % 100 == 0:
        torch.save(nca.state_dict(), "Trained_models/" + name + ".pth")

In [ ]:
plt.plot(log_iters, alpha_log, label='alpha')
plt.plot(log_iters, beta_log, label='beta')
plt.plot(log_iters, omega_log, label='omega')
plt.plot(log_iters, K_log, label='K')
plt.plot(log_iters, a, label='a')
plt.plot(log_iters, b, label='b')
plt.plot(log_iters, d, label='d')

In [ ]:
del nca
del optim
torch.cuda.empty_cache()

In [ ]:

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Convert your results to a DataFrame
df = pd.DataFrame(grid_results, columns=['Channels', 'GeneSize', 'Loss'])

# Save to CSV (Always a good idea for your thesis records!)
df.to_csv("grid_search_results.csv", index=False)


plt.figure(figsize=(10, 6))

# Sort to ensure lines are drawn correctly
df = df.sort_values(by=['GeneSize', 'Channels'])

# Create a line for each Gene Size configuration
for gene_size in sorted(df['GeneSize'].unique()):
    subset = df[df['GeneSize'] == gene_size]
    plt.plot(subset['Channels'], subset['Loss'], marker='o', label=f'Gene Size: {gene_size}')

plt.title('NCA Performance: Impact of Channels and Gene Size (lizard)', fontsize=14)
plt.xlabel('Number of Channels (C)', fontsize=12)
plt.ylabel('Final Loss (MSE)', fontsize=12)
plt.xticks(channel_configs)  # Show only the tested channel counts
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend(title="Complexity (G)")
plt.show()

In [ ]:
#@title Video Utils { vertical-output: true}
path_video = "Saved_frames/GeneCA"

if not os.path.exists(path_video):
    os.makedirs(path_video)
    print(f"Path: {path_video} created")
else:
    print(f"Path: {path_video} already exists, all OK!")

def write_frame(x, path, frame_number, height, width, chn):
  image_np = x.clone().detach().cpu().permute(0,2,3,1).numpy().clip(0,1)[0,:,:,:3]
  plt.imsave(f"{path}/frame_{frame_number}.png", image_np)

def make_video(path, total_frames, height, width):
    # 'VP80' is the standard codec for WebM files
    fourcc = cv2.VideoWriter_fourcc(*'VP80')

    # Save as .webm
    video_filename = os.path.join(path, 'output_video.webm')

    # Ensure (width, height) matches your frame dimensions exactly
    out = cv2.VideoWriter(video_filename, fourcc, 30.0, (width, height))

    for frame_number in range(total_frames):
        frame_path = os.path.join(path, f"frame_{frame_number}.png")
        frame = cv2.imread(frame_path)

        if frame is not None:
            # Critical: Resize if the image doesn't match the VideoWriter setup
            if (frame.shape[1], frame.shape[0]) != (width, height):
                frame = cv2.resize(frame, (width, height))

            out.write(frame)
        else:
            print(f"Warning: Could not read {frame_path}")

    out.release()
    print(f"WebM video saved to: {video_filename}")

def place_seed(x, center_x, center_y, seeds, seed_index):
    x[:,3:-8,center_x,center_y] = 1
    x[:,-4:,center_x,center_y] = seeds[seed_index]
    return x

In [ ]:
#@title Create Video { vertical-output: true}
#del nca
#del optim
torch.cuda.empty_cache()
seeds = []
seeds = [torch.tensor([0.0,0.0,0.0,1.0], device=DEVICE),torch.tensor([0.0,0.0,1.0,0.0], device=DEVICE),
         torch.tensor([0.0,1.0,0.0,0.0], device=DEVICE), torch.tensor([0.0,1.0,0.0,1.0], device=DEVICE),
         torch.tensor([0.0,1.0,1.0,0.0], device=DEVICE),torch.tensor([0.0,1.0,1.0,1.0], device=DEVICE), torch.tensor([0.0,0.0,1.0,1.0], device=DEVICE)]


# Spaced out to prevent shapes from growing into each other
seed_locs = [[20, 30], [20, 80], [20, 130],[80, 30], [80, 80], [80, 130],[130, 30]]

seed_index = 0

nca = GeneCA(CHANNELS,hidden_n=HIDDEN_SIZE, gene_size=GENE_COUNT)
nca.load_state_dict(torch.load("Trained_models/Gene/GeneCA_gene_size_3.pth"))
nca.to(DEVICE).eval()
x_prime = torch.zeros((1, CHANNELS, HEIGHT*4, WIDTH*4), device=DEVICE)


frame_count = 499
for i in range(frame_count):
    x_prime = nca(x_prime)
    if i % 50 == 0:
        place_seed(x_prime, seed_locs[seed_index][0], seed_locs[seed_index][1], seeds, seed_index)
        seed_index = (seed_index + 1) % len(seeds)
    x_prime = x_prime.detach()
    write_frame(x_prime, path_video, i, HEIGHT*4, WIDTH*4,CHANNELS)
make_video(path_video, frame_count, HEIGHT*4, WIDTH*4)


In [ ]:
#@title Display Vide { vertical-output: true}
Video(path_video+'/output_video.webm', embed=True, width=320, height=320)